# 构建复权价格序列

## 目标

得到一张同时包含原始收盘价、因子、锚点与复权收盘价的表；不计算策略收益。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"adjusted-price-series\",\"identity\":\"synthetic\",\"args\":[[{\"security\":\"DEMO\",\"date\":\"2025-01-02\",\"close\":100,\"factor\":1},{\"security\":\"DEMO\",\"date\":\"2025-01-03\",\"close\":50,\"factor\":2},{\"security\":\"DEMO\",\"date\":\"2025-01-06\",\"close\":51,\"factor\":2}],\"2025-01-06\"],\"expected\":[{\"security\":\"DEMO\",\"date\":\"2025-01-02\",\"close\":100,\"factor\":1,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":50},{\"security\":\"DEMO\",\"date\":\"2025-01-03\",\"close\":50,\"factor\":2,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":50},{\"security\":\"DEMO\",\"date\":\"2025-01-06\",\"close\":51,\"factor\":2,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":51}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 先冻结价格定义

确认输入是未复权价格，而不是已经复权的价格。Tushare日线说明中，close与除权后的pre_close具有不同含义；不要拿两者混算。保留供应商字段、币种和版本，先按证券、交易日查重。

### 2. 按同一交易日连接因子

以证券标识和交易日期精确连接价格与因子。缺失因子、重复键、非正价格或非正因子应停止计算，不能默认填1。停牌日没有成交记录并不等于价格为零；需要另存交易日历与缺失原因。

### 3. 声明归一化锚点

在采用乘法复权因子的前提下，示例用 adjusted_close = close × factor / anchor_factor。固定末日作为锚点时，末日复权价等于原价。更换锚点会改变整条序列的数值尺度，因此应把锚点写进结果，而不是只写“前复权”。

### 4. 检查跳变与单位

示例中的虚构股票从100变为50，同时因子从1变为2；归一化后两日均为50。这里只说明二拆一的机械调整，不代表真实公司行动。不要给成交量、成交额直接乘同一价格因子；它们需要各自的定义。

### 方法与假设

- 重复复权会制造错误跳变；已复权输入必须拒绝。
- 最新因子版本并不证明历史时点可得；回看研究要冻结因子快照。
- 复权价格不自动等于含税费、现金分红再投资的总收益序列。

In [ ]:
def adjust_prices(rows, anchor_date):
    """Normalize one synthetic price series to the selected factor anchor."""
    from datetime import date
    import math

    if not rows:
        raise ValueError("empty_input")
    seen = set()
    security = rows[0].get("security")
    for row in rows:
        if not security or row.get("security") != security:
            raise ValueError("one_security_required")
        day = row.get("date", "")
        try:
            valid_date = date.fromisoformat(day).isoformat() == day
        except (ValueError, TypeError):
            valid_date = False
        if not valid_date or day in seen:
            raise ValueError("invalid_or_duplicate_date")
        for field in ("close", "factor"):
            value = row.get(field)
            if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
                raise ValueError("invalid_price_or_factor")
        seen.add(day)
    anchor = next((row for row in rows if row["date"] == anchor_date), None)
    if anchor is None:
        raise ValueError("missing_anchor")
    return [dict(row, anchorDate=anchor_date,
                 adjustedClose=round(row["close"] * row["factor"] / anchor["factor"], 6))
            for row in sorted(rows, key=lambda item: item["date"])]


### 运行小样本

输出3行，adjustedClose依次为50、50、51；原始close仍为100、50、51。锚点价格不变，原始数据不被覆盖。

In [ ]:
result = adjust_prices(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `cn.equity.daily`
- `cn.dataset.adj_factor`

### 参考资料

- [Tushare：复权因子字段](https://tushare.pro/document/2?doc_id=28)
- [Tushare：日线价格口径](https://tushare.pro/document/2?doc_id=27)

[返回教程](https://tradingdatas.com/recipes/adjusted-price-series/)